In [2]:
# L5 English Conjunctions/Adverb/Adjectives Finder
# Core DFA logic and processing functions
# This code implements a Deterministic Finite Automaton (DFA) to recognize
# specific English conjunctions, adverbs, and adjectives: "and", "most", "good",
# "bad", "pretty", "blue", "dirty". It processes text character by character,
# generates reports, and creates DFA diagrams for visualization.

import os
import tkinter as tk
from tkinter import filedialog, scrolledtext, messagebox, Toplevel, Label, Canvas, Scrollbar, Frame, Button
import platform
import subprocess
try:
    from PIL import Image, ImageTk
    PIL_AVAILABLE = True        # Flag to check if Pillow is available for image handling
except ImportError:
    PIL_AVAILABLE = False
try:
    import graphviz
    GRAPHVIZ_AVAILABLE = True
except ImportError:
    GRAPHVIZ_AVAILABLE = False

# List of keywords restricted to English conjunctions, adverbs, and adjectives
# This list defines the words the DFA will recognize
keywords = ['and', 'most', 'good', 'bad', 'blue', 'pretty', 'dirty']

# Define acceptable categories for validation
# These sets ensure the program only accepts valid conjunctions, adverbs, and adjectives
ACCEPTABLE_CATEGORIES = {
    'conjunctions': {'and'},
    'adverbs': {'most'},
    'adjectives': {'good', 'bad', 'blue', 'pretty', 'dirty'}
}

def validate_keywords(keywords):
    """Validate that all keywords are conjunctions, adverbs, or adjectives.
    Raises ValueError if a keyword is not in ACCEPTABLE_CATEGORIES."""
    for word in keywords:
        if word.lower() not in ACCEPTABLE_CATEGORIES['conjunctions'] and \
           word.lower() not in ACCEPTABLE_CATEGORIES['adverbs'] and \
           word.lower() not in ACCEPTABLE_CATEGORIES['adjectives']:
            raise ValueError(f"Keyword '{word}' is not a recognized conjunction, adverb, or adjective.")

# Validate keywords before generating DFA
validate_keywords(keywords)

def generate_dfa(keywords):
    """Generate a Deterministic Finite Automaton (DFA) transition table and accepting states.
    - Input: List of keywords (conjunctions, adverbs, adjectives).
    - Output: Tuple of (transitions dictionary, accepting_states dictionary).
    - Process: Creates states for each character in each keyword, with a 'trap' state for invalid inputs."""
    transitions = {0: {}, 'trap': {}}  # Initialize with start state 0 and trap state
    accepting_states = {}              # Maps accepting states to their corresponding keywords 
    state_counter = 1                  # Counter for new states
    prefix_states = {(0, ''): 0}       # Tracks states for prefix paths to avoid redundancy 

#     For each character in a word, it adds a path in the DFA and connects states together.
    for word in keywords:
        current_state = 0              # Start at initial state for each keyword
        word = word.lower()            # Convert to lowercase for case-insensitive matching 
        for i, char in enumerate(word):
            prefix = word[:i + 1]      # Build prefix of the word up to current character
            prefix_tuple = (current_state, char)
            if prefix_tuple in prefix_states:
                next_state = prefix_states[prefix_tuple]  # Reuse existing state if prefix exists
            else:
                next_state = state_counter                   # Create new state
                prefix_states[prefix_tuple] = next_state
                transitions[next_state] = {}                 # Initialize new state's transitions
                state_counter += 1
            transitions[current_state][char] = next_state  # Add transition for current character 
            current_state = next_state                    # Move to next state
        accepting_states[current_state] = word           # Mark the final state as accepting

    # Add trap transitions for all lowercase letters to handle invalid inputs
    for state in transitions:
        for char in 'abcdefghijklmnopqrstuvwxyz':
            if char not in transitions[state]:
                transitions[state][char] = 'trap'

    return transitions, accepting_states

# Generate the DFA based on the validated keywords
transitions, accepting_states = generate_dfa(keywords)

# Highlight settings for GUI display
highlight_fg = 'black'              # Foreground color for highlighted text
highlight_bg = 'yellow'             # Background color for highlighted keywords
status_colors = {
    'ACCEPTED': '#008000',          # Green for successful matches
    'REJECTED': '#ff0000'           # Red for no matches
}

def is_boundary(char, text, pos):
    """Check if a character is a word boundary (space, punctuation, or text edge).
    Returns True if the character marks the start/end of a standalone word."""
    if char is None or pos < 0 or pos >= len(text):
        return True
    return char in ' \n.,;!?()[]{}":'

def process_text(text):
    """Process input text using the DFA to find occurrences of keywords.
    - Input: Text string to analyze.
    - Output: List of (word, position) tuples for matched keywords.
    - Logic: Scans text character by character, following DFA transitions and checking boundaries."""
    text_lower = text.lower()          # Convert to lowercase for case-insensitive matching 
    current_state = 0                  # Start at initial DFA state
    occurrences = []                   # Store matched (word, position) pairs
    word_start = 0                     # Track the start position of the current word
    pos = 0                            # Current position in the text

    while pos < len(text_lower):
        char = text_lower[pos]         # Get the current character
        next_state = transitions.get(current_state, {}).get(char, 'trap')  # Follow DFA transition
        if next_state in accepting_states:  # Check if current state is accepting 
            next_char = text_lower[pos + 1] if pos + 1 < len(text_lower) else None
            prev_char = text_lower[word_start - 1] if word_start > 0 else None
            is_next_boundary = is_boundary(next_char, text_lower, pos + 1) # Make sure the word is not part of another word, then save it.
            is_prev_boundary = is_boundary(prev_char, text_lower, word_start - 1)
            if is_next_boundary and is_prev_boundary:
                word = accepting_states[next_state]  # Get the matched keyword
                occurrences.append((word, word_start))  # Record the match
                current_state = 0                    # Reset to initial state
                word_start = pos + 1                 # Move word start past the match
            else:
                current_state = next_state           # Continue with next state if no boundary
        elif next_state == 'trap':         # Reset if invalid transition 
            current_state = 0
            word_start = pos + 1
        else:
            current_state = next_state      # Move to next state
            if current_state == 0:
                word_start = pos + 1         # Update word start if back to initial state
        pos += 1                           # Move to next character

    return occurrences

def create_dfa_graph(keyword):
    """Create a Graphviz diagram for a specific keyword's DFA.
    - Input: A keyword from the keywords list.
    - Output: Graphviz Digraph object representing the DFA.
    - Purpose: Visualizes state transitions for presentation and debugging."""
    dot = graphviz.Digraph(comment=f'DFA for {keyword}')
    dot.attr(rankdir='LR', size='10,6', fontsize='12')  # Left-to-right layout

    # Track states and transitions for the keyword
    states = []
    keyword_transitions = []
    state = 0
    word_so_far = ""
    states.append((state, word_so_far))

    for char in keyword:
        next_state = transitions[state].get(char)
        if next_state is None:
            break
        word_so_far += char
        states.append((next_state, word_so_far))
        keyword_transitions.append((state, char, next_state))
        state = next_state

    # Add trap state for invalid transitions
    states.append(('trap', 'trap'))

    # Draw states in the diagram
    for state, label in states:
        if state == 'trap':
            dot.node('trap', 'Trap', shape='circle', style='dotted', color='gray', fontsize='12')
        elif state in accepting_states:
            dot.node(str(state), f"q{state}\n{label.upper()}", shape='doublecircle', color='green', fontsize='12')
        else:
            dot.node(str(state), f"q{state}\n{label}", shape='circle', fontsize='12')

    # Add start arrow to q0 (state 0)
    dot.node('start', '', shape='none')
    dot.edge('start', '0')

    # Draw transitions between states
    for from_state, char, to_state in keyword_transitions:
        dot.edge(str(from_state), str(to_state), label=char, fontsize='12')
        # Add trap transitions for unhandled characters
        handled_char = set(char)
        unhandled_chars = set('abcdefghijklmnopqrstuvwxyz') - handled_char
        if unhandled_chars and from_state != 0:
            dot.edge(str(from_state), 'trap', label='other', fontsize='12')

    # Trap self-loop for any character
    dot.edge('trap', 'trap', label='any', fontsize='12')

    return dot

def generate_diagrams():
    """Generate and save DFA diagrams for each keyword.
    - Output: List of file paths to generated PNG images.
    - Note: Requires Graphviz and Pillow; skips if dependencies are missing."""
    keywords = list(accepting_states.values())
    diagram_dir = "diagrams"
    os.makedirs(diagram_dir, exist_ok=True)  # Create directory if it doesn't exist
    image_paths = []

    for keyword in keywords:
        dot = create_dfa_graph(keyword)
        filepath = os.path.join(diagram_dir, f"dfa_{keyword}")
        try:
            dot.render(filepath, format='png', cleanup=True)  # Render as PNG
            image_paths.append(f"{filepath}.png")
        except Exception as e:
            print(f"Failed to render diagram for {keyword}: {e}")
            image_paths.append(None)

    return image_paths

def generate_report(text, occurrences, report_file="dfa_report.txt"):
    """Generate a detailed report of DFA matches and save it to a file.
    - Input: Text input and list of (word, position) occurrences.
    - Output: File path of the saved report.
    - Includes: Introduction, implementation details, match summary, and DFA structure."""
    total_matches = len(occurrences)
    keywords = list(accepting_states.values())
    
    report_lines = ["--- DFA RECOGNIZER REPORT ---"]
    
    report_lines.append("\nI. Introduction")
    report_lines.append("This program implements a Deterministic Finite Automaton (DFA) for the L5 language, recognizing English conjunctions, adverbs, and adjectives: " + ", ".join(keywords) + ".")
    report_lines.append("The DFA processes text character by character from left to right, identifying standalone words with proper boundaries, as required by the CPT 411 assignment.")
    
    report_lines.append("\nII. Implementation Information")
    report_lines.append("a. String Processing:")
    report_lines.append("- Input text is read from a file or entered manually via a Tkinter GUI.")
    report_lines.append("- Text is converted to lowercase for case-insensitive matching.")
    report_lines.append("- The DFA processes one character at a time, using a transition table to track states.")
    report_lines.append("- Word boundaries (spaces, punctuation, or text edges) are checked using the is_boundary function.")
    report_lines.append("b. Programming Constructs:")
    report_lines.append("- Python dictionaries for DFA transitions and accepting states, generated automatically.")
    report_lines.append("- Tkinter for the GUI, displaying input text, occurrences, highlighted matches, and status.")
    report_lines.append("- Graphviz (optional) for DFA diagram visualization.")
    report_lines.append("- Error handling for file loading and missing dependencies (Pillow, Graphviz).")
    
    report_lines.append("\nInput Text:")
    report_lines.append(text + "\n")
    
    for keyword in keywords:
        keyword_occ = [(word, pos) for word, pos in occurrences if word == keyword]
        positions = [pos for _, pos in keyword_occ]
        status = "ACCEPTED" if positions else "REJECTED"
        report_lines.append(f"\nKeyword: '{keyword}'")
        report_lines.append(f"Status: {status}")
        report_lines.append(f"Occurrences: {len(positions)} at positions: {positions}")
        if positions:
            report_lines.append("Snippets:")
            for pos in positions:
                start = max(0, pos - 20)
                end = min(len(text), pos + len(keyword) + 20)
                snippet = text[start:pos] + f">>>{text[pos:pos+len(keyword)]}<<<" + text[pos+len(keyword):end]
                report_lines.append(f"  ...{snippet}...")
    report_lines.append(f"\n✅ Total Matches: {total_matches}\n")
    
    for keyword in keywords:
        report_lines.append(f"=== DFA Structure for '{keyword}' ===")
        state = 0
        states = [f"q{state}"]
        trans_list = []
        for char in keyword:
            next_state = transitions[state].get(char)
            if next_state is None:
                break
            states.append(f"q{next_state}")
            trans_list.append(f"  q{state} --{char}--> q{next_state}")
            state = next_state
        report_lines.append(f"States: {', '.join(states)}")
        report_lines.append("Transitions:")
        for t in trans_list:
            report_lines.append(t)
        report_lines.append(f"  q{state} is the accepting state.\n")
    
    report_lines.append("\nIII. Conclusion")
    report_lines.append("The DFA successfully recognizes the specified English conjunctions, adverbs, and adjectives in the input text, providing a user-friendly GUI for interaction and visualization. The program supports both sample and new text inputs, with highlighted matches and DFA diagrams for demonstration.")
    
    report_content = "\n".join(report_lines)
    print(report_content)
    
    try:
        with open(report_file, 'w', encoding='utf-8') as f:
            f.write(report_content)
        return report_file
    except Exception as e:
        print(f"Failed to save report: {e}")
        return None

class L5FinderApp:
    """A GUI application to recognize conjunctions, adverbs, and adjectives using a DFA.
    Provides an interactive interface for text input, processing, and visualization."""
    
    def __init__(self, root):
        """Initialize the GUI application with all components.
        Sets up the main window, canvas, scrollbar, and various frames for input/output."""
        self.root = root
        self.root.title("L5 DFA Recognizer - Conjunctions, Adverbs, Adjectives")
        self.root.geometry("900x700")  # Fixed width for consistency
        self.root.configure(bg="#f0f4f7")

        # Create a scrollable canvas for the entire interface
        self.canvas = Canvas(self.root, bg="#f0f4f7")
        self.scrollbar = Scrollbar(self.root, orient="vertical", command=self.canvas.yview)
        self.scrollable_frame = Frame(self.canvas, bg="#f0f4f7")

        def configure_canvas(event):
            """Adjust canvas width to match window and update scroll region."""
            canvas_width = event.width
            self.canvas.itemconfig(self.canvas_window, width=canvas_width)
            self.canvas.configure(scrollregion=self.canvas.bbox("all"))

        self.canvas.bind("<Configure>", configure_canvas)
        self.canvas_window = self.canvas.create_window((0, 0), window=self.scrollable_frame, anchor="nw")
        self.canvas.configure(yscrollcommand=self.scrollbar.set)
        self.scrollable_frame.bind("<Configure>", lambda e: self.canvas.configure(scrollregion=self.canvas.bbox("all")))
        self.canvas.pack(side="left", fill="both", expand=True)
        self.scrollbar.pack(side="right", fill="y")

        self.main_frame = tk.Frame(self.scrollable_frame, bg="#ffffff", bd=2, relief="groove")
        self.main_frame.pack(padx=30, pady=30, fill="both", expand=True)

        # Store recognized keywords for display and searching
        self.patterns = list(accepting_states.values())
        
        # Title and interface components
        self.title_label = tk.Label(self.main_frame, text="DFA Recognizer", font=("Segoe UI", 18, "bold"), bg="#ffffff", fg="#2c3e50")
        self.title_label.pack(pady=10)

        self.file_frame = tk.Frame(self.main_frame, bg="#ffffff")
        self.file_frame.pack(pady=10)
        self.file_label = tk.Label(self.file_frame, text="No file selected", font=("Segoe UI", 12), bg="#ffffff", fg="#2c3e50")
        self.file_label.pack(side=tk.LEFT, padx=5)
        self.browse_btn = tk.Button(self.file_frame, text="📂 Load File", command=self.load_file, font=("Segoe UI", 12, "bold"), bg="#3498db", fg="white", width=15, relief="flat", activebackground="#2980b9")
        self.browse_btn.pack(side=tk.LEFT, padx=5)

        self.btn_frame = tk.Frame(self.main_frame, bg="#ffffff")
        self.btn_frame.pack(pady=10)
        self.process_btn = tk.Button(self.btn_frame, text="▶ Process", command=self.process, font=("Segoe UI", 12, "bold"), bg="#3498db", fg="white", width=12, relief="flat", activebackground="#2980b9")
        self.process_btn.pack(side=tk.LEFT, padx=5)
        self.clear_btn = tk.Button(self.btn_frame, text="🗑 Clear", command=self.clear, font=("Segoe UI", 12, "bold"), bg="#e74c3c", fg="white", width=12, relief="flat", activebackground="#c0392b")
        self.clear_btn.pack(side=tk.LEFT, padx=5)
        self.dfa_btn = tk.Button(self.btn_frame, text="📊 Show DFA Diagrams", command=self.show_dfa_diagrams, font=("Segoe UI", 12, "bold"), bg="#2ecc71", fg="white", width=22, relief="flat", activebackground="#27ae60")
        self.dfa_btn.pack(side=tk.LEFT, padx=5)
        self.report_btn = tk.Button(self.btn_frame, text="📜 Generate Report", command=self.generate_report_file, font=("Segoe UI", 12, "bold"), bg="#f39c12", fg="white", width=20, relief="flat", activebackground="#e67e22")
        self.report_btn.pack(side=tk.LEFT, padx=5)

        self.search_frame = tk.Frame(self.main_frame, bg="#ffffff")
        self.search_frame.pack(pady=5)
        self.search_label = tk.Label(self.search_frame, text="Search Keyword:", font=("Segoe UI", 12), bg="#ffffff", fg="#2c3e50")
        self.search_label.pack(side=tk.LEFT, padx=5)
        self.search_entry = tk.Entry(self.search_frame, font=("Segoe UI", 12), relief="flat", highlightthickness=1, highlightbackground="#d1d4d7")
        self.search_entry.pack(side=tk.LEFT, padx=5, fill="x", expand=True)
        self.search_btn = tk.Button(self.search_frame, text="🔍 Search", command=self.search_keyword, font=("Segoe UI", 12, "bold"), bg="#e91e63", fg="white", width=12, relief="flat", activebackground="#c2185b")
        self.search_btn.pack(side=tk.LEFT, padx=5)

        patterns_text = "Patterns: " + ", ".join(self.patterns)
        self.patterns_label = tk.Label(self.main_frame, text=patterns_text, font=("Segoe UI", 12, "bold"), bg="#ffffff", fg="#420b80")
        self.patterns_label.pack(pady=5, fill="x")

        self.status_label = tk.Label(self.main_frame, text="Status: None", font=("Segoe UI", 12, "bold"), bg="#ffffff", fg="#2c3e50")
        self.status_label.pack(pady=5, fill="x")

        self.input_label = tk.Label(self.main_frame, text="Input Text (Type or Load File)", font=("Segoe UI", 12, "bold"), bg="#ffffff", fg="#2c3e50")
        self.input_label.pack(pady=5, fill="x")
        tk.Frame(self.main_frame, height=2, bg="#3498db").pack(fill="x", padx=15)

        self.input_text = scrolledtext.ScrolledText(self.main_frame, wrap=tk.WORD, height=10, font=("Courier New", 12), bg="#ffffff", borderwidth=2, relief="flat", highlightthickness=1, highlightbackground="#d1d4d7")
        self.input_text.pack(padx=15, pady=5, fill="both", expand=True)

        self.occ_label = tk.Label(self.main_frame, text="Occurrences", font=("Segoe UI", 12, "bold"), bg="#ffffff", fg="#2c3e50")
        self.occ_label.pack(pady=5, fill="x")
        tk.Frame(self.main_frame, height=2, bg="#3498db").pack(fill="x", padx=15)

        self.occ_text = scrolledtext.ScrolledText(self.main_frame, wrap=tk.WORD, height=12, font=("Courier New", 12), bg="#ffffff", borderwidth=2, relief="flat", highlightthickness=1, highlightbackground="#d1d4d7")
        self.occ_text.pack(padx=15, pady=5, fill="both", expand=True)
        self.occ_text.config(state='disabled')

        self.highlight_label = tk.Label(self.main_frame, text="Highlighted Text", font=("Segoe UI", 12, "bold"), bg="#ffffff", fg="#2c3e50")
        self.highlight_label.pack(pady=5, fill="x")
        tk.Frame(self.main_frame, height=2, bg="#3498db").pack(fill="x", padx=15)

        self.highlight_text = scrolledtext.ScrolledText(self.main_frame, wrap=tk.WORD, height=15, font=("Courier New", 12), bg="#ffffff", borderwidth=2, relief="flat", highlightthickness=1, highlightbackground="#d1d4d7")
        self.highlight_text.pack(padx=15, pady=5, fill="both", expand=True)
        self.highlight_text.config(state='disabled')

        self.highlight_text.tag_configure('keyword', foreground=highlight_fg, background=highlight_bg, font=("Courier New", 12, "bold"))
        self.highlight_text.tag_configure('search_highlight', foreground='black', background='#ff9900', font=("Courier New", 12, "bold"))
        self.occ_text.tag_configure('search_highlight', foreground='black', background='#ff9900', font=("Courier New", 12, "bold"))

        self.text_content = ""  # Store the current input text
        self.images = []
        self.root.update()

        self.canvas.bind_all("<MouseWheel>", self._on_mousewheel)

    def _on_mousewheel(self, event):
        """Enable mouse wheel scrolling for the canvas."""
        self.canvas.yview_scroll(int(-1 * (event.delta / 120)), "units")

    def load_file(self):
        """Load a text file and display its content in the input text area.
        Handles file selection and error cases like empty files."""
        file_path = filedialog.askopenfilename(filetypes=[("Text files", "*.txt")])
        if file_path:
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    self.text_content = f.read()
                if not self.text_content.strip():
                    raise ValueError("File is empty.")
                self.file_label.config(text=os.path.basename(file_path))
                self.input_text.delete(1.0, tk.END)
                self.input_text.insert(tk.END, self.text_content)
            except Exception as e:
                messagebox.showerror("Error", f"Failed to load file: {e}")

    def process(self):
        """Process the loaded or entered text and display results.
        Uses the DFA to find matches and updates the GUI with occurrences and highlights."""
        self.text_content = self.input_text.get(1.0, tk.END).strip()
        if not self.text_content:
            messagebox.showwarning("Warning", "Please enter text or load a file.")
            return
        
        try:
            occurrences = process_text(self.text_content)
            total_matches = len(occurrences)
            status = f"{total_matches} match(es) found! ACCEPTED" if occurrences else "No matches found. REJECTED"
            
            self.status_label.config(text=f"Status: {status}", fg=status_colors['ACCEPTED' if occurrences else 'REJECTED'], font=("Segoe UI", 12, "bold"))
            self.occ_text.config(state='normal')
            self.occ_text.delete(1.0, tk.END)
            for pattern in sorted(set(word for word, _ in occurrences)):
                entries = [(word, pos) for word, pos in occurrences if word == pattern]
                self.occ_text.insert(tk.END, f"Pattern: \"{pattern}\"\n")
                self.occ_text.insert(tk.END, f"Occurrences: {len(entries)}\n")
                self.occ_text.insert(tk.END, f"Positions: {[pos for _, pos in entries]}\n\n")
            self.occ_text.config(state='disabled')
            
            self.highlight_text.config(state='normal')
            self.highlight_text.delete(1.0, tk.END)
            current_pos = 0
            for word, pos in sorted(occurrences, key=lambda x: x[1]):
                self.highlight_text.insert(tk.END, self.text_content[current_pos:pos])
                original_word = self.text_content[pos:pos+len(word)]
                self.highlight_text.insert(tk.END, original_word, 'keyword')
                current_pos = pos + len(word)
            self.highlight_text.insert(tk.END, self.text_content[current_pos:])
            self.highlight_text.config(state='disabled')
            
            generate_report(self.text_content, occurrences)
            self.canvas.configure(scrollregion=self.canvas.bbox("all"))
            
        except Exception as e:
            messagebox.showerror("Error", f"Processing failed: {e}")

    def clear(self):
        """Clear all fields and reset the application to its initial state."""
        self.text_content = ""
        self.file_label.config(text="No file selected")
        self.status_label.config(text="Status: None", fg="#2c3e50", font=("Segoe UI", 12, "bold"))
        self.input_text.delete(1.0, tk.END)
        self.occ_text.config(state='normal')
        self.occ_text.delete(1.0, tk.END)
        self.occ_text.config(state='disabled')
        self.highlight_text.config(state='normal')
        self.highlight_text.delete(1.0, tk.END)
        self.highlight_text.config(state='disabled')
        self.search_entry.delete(0, tk.END)
        self.canvas.configure(scrollregion=self.canvas.bbox("all"))

    def show_dfa_diagrams(self):
        """Show DFA diagrams for all keywords in a new window with zoom functionality.
        Requires Graphviz and Pillow; displays error if dependencies are missing."""
        if not PIL_AVAILABLE:
            messagebox.showerror("Error", "Pillow is not installed. Install with 'pip install Pillow'.")
            return
        if not GRAPHVIZ_AVAILABLE:
            messagebox.showerror("Error", "Graphviz is not installed. Install with 'pip install graphviz' and ensure Graphviz is in PATH.")
            return

        try:
            image_paths = generate_diagrams()
            top = Toplevel(self.root)
            top.title("DFA Diagrams for All Keywords")
            top.geometry("900x700")

            canvas = Canvas(top, bg="#f0f4f7")
            scrollbar = Scrollbar(top, orient="vertical", command=canvas.yview)
            scrollable_frame = Frame(canvas, bg="#f0f4f7")

            def configure_diagram_canvas(event):
                canvas_width = event.width
                canvas.itemconfig(canvas_window, width=canvas_width)
                canvas.configure(scrollregion=canvas.bbox("all"))

            canvas.bind("<Configure>", configure_diagram_canvas)
            canvas_window = canvas.create_window((0, 0), window=scrollable_frame, anchor="nw")
            canvas.configure(yscrollcommand=scrollbar.set)
            scrollable_frame.bind("<Configure>", lambda e: canvas.configure(scrollregion=canvas.bbox("all")))
            canvas.pack(side="left", fill="both", expand=True)
            scrollbar.pack(side="right", fill="y")

            zoom_frame = Frame(top, bg="#f0f4f7")
            zoom_frame.pack(pady=5, fill="x")
            zoom_in_btn = Button(zoom_frame, text="+", command=lambda: self.zoom_canvas(canvas, 1.2))
            zoom_in_btn.pack(side=tk.LEFT, padx=5)
            zoom_out_btn = Button(zoom_frame, text="-", command=lambda: self.zoom_canvas(canvas, 0.8))
            zoom_out_btn.pack(side=tk.LEFT, padx=5)

            self.images = []
            for i, (keyword, img_path) in enumerate(zip(self.patterns, image_paths)):
                frame = Frame(scrollable_frame, bg="#ffffff", bd=2, relief="groove")
                frame.pack(pady=10, padx=10, fill="x")
                label = Label(frame, text=f"DFA for '{keyword}'", font=("Segoe UI", 12, "bold"), bg="#ffffff")
                label.pack(pady=5)
                if img_path and os.path.exists(img_path):
                    img = Image.open(img_path)
                    img_width = 850
                    aspect_ratio = img.height / img.width
                    img_height = int(img_width * aspect_ratio)
                    img = img.resize((img_width, img_height), Image.LANCZOS)
                    photo = ImageTk.PhotoImage(img)
                    img_label = Label(frame, image=photo, bg="#ffffff")
                    img_label.image = photo
                    img_label.pack(fill="x")
                    self.images.append((photo, img, img_label))
                else:
                    error_label = Label(frame, text="Diagram not available", font=("Segoe UI", 12), bg="#ffffff", fg="red")
                    error_label.pack()
            canvas.bind("<MouseWheel>", lambda e: self.zoom_canvas(canvas, 1.1 if e.delta > 0 else 0.9))

        except Exception as e:
            messagebox.showerror("Error", f"Error displaying DFA diagrams: {e}")

    def zoom_canvas(self, canvas, scale):
        """Zoom the canvas by scaling all images within a specified range (100-2000 pixels)."""
        for photo, img, img_label in self.images:
            current_width = img.width
            current_height = img.height
            new_width = int(current_width * scale)
            new_height = int(current_height * scale)
            if 100 <= new_width <= 2000:
                img_resized = img.resize((new_width, new_height), Image.LANCZOS)
                new_photo = ImageTk.PhotoImage(img_resized)
                img_label.configure(image=new_photo)
                img_label.image = new_photo
        canvas.configure(scrollregion=canvas.bbox("all"))

    def search_keyword(self):
        """Search for a keyword in the highlighted text and occurrences, scroll to first occurrence.
        Validates the keyword against the patterns list and updates the GUI accordingly."""
        keyword = self.search_entry.get().strip().lower()
        if not keyword:
            messagebox.showwarning("Warning", "Please enter a keyword to search.")
            return
        if keyword not in self.patterns:
            messagebox.showwarning("Warning", f"'{keyword}' is not a recognized keyword. Choose from: {', '.join(self.patterns)}")
            return
        if not self.text_content:
            messagebox.showwarning("Warning", "Please process some text first.")
            return

        try:
            occurrences = process_text(self.text_content)
            keyword_occurrences = [(word, pos) for word, pos in occurrences if word == keyword]
            if not keyword_occurrences:
                messagebox.showinfo("Info", f"No occurrences of '{keyword}' found.")
                return

            self.highlight_text.config(state='normal')
            self.highlight_text.tag_remove('search_highlight', '1.0', tk.END)
            self.highlight_text.delete(1.0, tk.END)
            current_pos = 0
            for word, pos in sorted(occurrences, key=lambda x: x[1]):
                self.highlight_text.insert(tk.END, self.text_content[current_pos:pos])
                original_word = self.text_content[pos:pos+len(word)]
                tag = 'search_highlight' if word == keyword else 'keyword'
                self.highlight_text.insert(tk.END, original_word, tag)
                current_pos = pos + len(word)
            self.highlight_text.insert(tk.END, self.text_content[current_pos:])
            self.highlight_text.config(state='disabled')
            first_pos = keyword_occurrences[0][1]
            mark = f"1.0 + {first_pos} chars"
            self.highlight_text.see(mark)

            self.occ_text.config(state='normal')
            self.occ_text.tag_remove('search_highlight', '1.0', tk.END)
            self.occ_text.delete(1.0, tk.END)
            line_number = 1
            keyword_line_start = None
            for pattern in sorted(set(word for word, _ in occurrences)):
                entries = [(word, pos) for word, pos in occurrences if word == pattern]
                pattern_text = f"Pattern: \"{pattern}\"\n"
                occ_text = f"Occurrences: {len(entries)}\n"
                pos_text = f"Positions: {[pos for _, pos in entries]}\n\n"
                if pattern == keyword:
                    keyword_line_start = line_number
                    self.occ_text.insert(tk.END, pattern_text, 'search_highlight')
                    self.occ_text.insert(tk.END, occ_text, 'search_highlight')
                    self.occ_text.insert(tk.END, pos_text, 'search_highlight')
                else:
                    self.occ_text.insert(tk.END, pattern_text + occ_text + pos_text)
                line_number += len(pattern_text.split('\n')) + len(occ_text.split('\n')) + len(pos_text.split('\n')) - 2
            self.occ_text.config(state='disabled')
            if keyword_line_start:
                self.occ_text.see(f"{keyword_line_start}.0")

            messagebox.showinfo("Success", f"Found {len(keyword_occurrences)} occurrence(s) of '{keyword}'. Scrolled to first match in both highlighted text and occurrences.")
            self.canvas.configure(scrollregion=self.canvas.bbox("all"))

        except Exception as e:
            messagebox.showerror("Error", f"Search failed: {e}")

    def generate_report_file(self):
        """Generate the report file and open it with the default system viewer."""
        if not self.text_content:
            messagebox.showwarning("Warning", "Please process some text first.")
            return
        occurrences = process_text(self.text_content)
        report_file = generate_report(self.text_content, occurrences)
        if report_file:
            try:
                if platform.system() == "Windows":
                    os.startfile(report_file)
                elif platform.system() == "Darwin":
                    subprocess.run(["open", report_file])
                else:
                    subprocess.run(["xdg-open", report_file])
                messagebox.showinfo("Success", f"Report saved to {report_file} and opened.")
            except Exception as e:
                messagebox.showerror("Error", f"Failed to open report file: {e}")

if __name__ == "__main__":
    root = tk.Tk()
    app = L5FinderApp(root)
    root.mainloop()